In [0]:
%pip uninstall -y psycopg2 psycopg2-binary

In [0]:
%pip install -q 'databricks-sdk>=0.118.0' sentence-transformers
dbutils.library.restartPython()


In [0]:
# Define las variables de entorno y haz los imports después del último reinicio de Python.

import os

os.environ["LAKEBASE_HOST"] = "ep-sparkling-smoke-d87sujf7.database.us-east-2.cloud.databricks.com"
os.environ["LAKEBASE_PORT"] = "5432"
os.environ["LAKEBASE_DB"] = "databricks_postgres"
os.environ["LAKEBASE_USER"] = "felixemilio9312@gmail.com"
os.environ["LAKEBASE_ENDPOINT"] = "projects/research-copilot/branches/production/endpoints/primary"
os.environ["HF_HOME"] = "/tmp/.cache/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/tmp/.cache/huggingface"

from src.embeddings.embed_papers import embed_pending_papers
embed_pending_papers()

In [0]:
import json
from src.db.connection import get_connection
from src.embeddings.embed_papers import embed_query
 
vec = embed_query("aprender los fundamentos de RAG en LLMs")
 
with get_connection() as conn, conn.cursor() as cur:
    cur.execute("SELECT * FROM search_papers(%s::vector, %s)", (json.dumps(vec), 5))
    for row in cur.fetchall():
        print(row["title"])

In [0]:
from src.db.connection import get_connection

with get_connection() as conn, conn.cursor() as cur:
    cur.execute("SELECT count(*) AS n FROM papers")
    print("Total papers:", cur.fetchone()["n"])

    cur.execute("SELECT count(*) AS n FROM papers WHERE embedding IS NOT NULL")
    print("Papers con embedding:", cur.fetchone()["n"])

    cur.execute("SELECT count(*) AS n FROM papers WHERE abstract IS NULL OR abstract = ''")
    print("Papers sin abstract (se saltan al embeber):", cur.fetchone()["n"])

    cur.execute("SELECT paper_id, title FROM papers WHERE embedding IS NOT NULL LIMIT 3")
    print("Muestra con embedding:", cur.fetchall())

    cur.execute("""
        SELECT proname, pg_get_function_identity_arguments(oid) AS args
        FROM pg_proc
        WHERE proname = 'search_papers'
    """)
    print("Firma de search_papers en Postgres:", cur.fetchall())

In [0]:
from src.embeddings.embed_papers import embed_pending_papers
embed_pending_papers(batch_size=50)